In [12]:
import pickle
import json
import time

import pandas as pd
from pyboolnet.external.bnet2primes import bnet_file2primes
from pystablemotifs.format import primes2bnet

from boolmore.eval.constraint import check_node
from boolmore.core.conversions import prime2rr

In [10]:
input_csv = "confidence_rank_ultrametric.csv"
output_bnet = "Tcell_merged_ultrametric.bnet"

json_file = "../case_study/T_cell/Tcell_config.json"
CACHE_FILE = "Tcell_primes_ultrametric.pkl"

In [3]:
# Load the CSV
df = pd.read_csv(input_csv)

required_columns = {"Regulator", "Target", "Sign"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Collect all nodes
all_nodes = set(df["Regulator"]) | set(df["Target"])

print("Number of nodes:", len(all_nodes))

Number of nodes: 106


In [4]:
# Group regulations by target
rules = {}

for _, row in df.iterrows():
    regulator = row["Regulator"]
    target = row["Target"]
    sign = str(row["Sign"]).strip().lower()

    if sign == "positive":
        literal = regulator
    elif sign == "negative":
        literal = f"!{regulator}"
    else:
        raise ValueError(
            f"Invalid sign '{row['Sign']}' for edge {regulator} -> {target}"
        )

    rules.setdefault(target, []).append(literal)

print("Number of rules:", len(rules))

Number of rules: 83


In [5]:
# Write the bnet file
with open(output_bnet, "w") as f:
    for node in sorted(all_nodes):
        if node in rules:
            rule = " & ".join(rules[node])
        else:
            # Source node
            rule = node
        f.write(f"{node},\t{rule}\n")

In [6]:
primes = bnet_file2primes(output_bnet)

In [7]:
with open(CACHE_FILE, "wb") as f:
    pickle.dump(primes, f)
print("Computed and cached primes.")

Computed and cached primes.


In [13]:
bnet = primes2bnet(primes)
print(bnet)

APC,            APC
BCL6,           STAT1&STAT1_2&STAT3&STAT4&!STAT5&!TBET&!TBET_2&!TGFB
CD28,           APC
CD4,            NOTCH1&!RUNX3&THPOK
CD8,            NOTCH1&RUNX3&!THPOK
CMAF,           STAT3
DLL1,           DLL1
EOMES,          RUNX3
FOXP3,          NFAT&SMAD2&SMAD3&STAT5&STAT5_2&!STAT6&!TBET&!TBET_2
GATA3,          !BCL6&!FOXP3&GATA3&IL25R&!IL29R&NFAT&!PU1&STAT5&STAT5_2&STAT6&!TBET&!TBET_2
GZMB,           EOMES
IFNAR,          IFNA_e&IFNB_e
IFNA_e,         IFNA_e
IFNBR,          IFNB_e
IFNB_e,         IFNB_e
IFNG,           !BCL6&EOMES&!GATA3&IL18R&!IL9&IRAK&NFAT&NFKB&RUNX3&STAT4&TBET&TBET_2&proliferation
IFNGR,          IFNG&IFNG_2&IFNG_e
IFNGR_2,        IFNG&IFNGR&IFNG_2&IFNG_e
IFNG_2,         !BCL6&EOMES&!GATA3&IFNG&IL18R&!IL9&IRAK&NFAT&NFKB&RUNX3&STAT4&TBET&TBET_2&proliferation
IFNG_e,         IFNG_e
IKB,            !TCR
IL10,           CMAF&IRF1&NFAT&STAT3&proliferation
IL10R,          IL10&IL10_e
IL10_e,         IL10_e
IL12R,          IL12RB1_2&IL12RB2&IL12_e
IL12RB1

In [8]:
regulators_dict = {}
rr_dict = {}
signs_dict = {}

for node in primes:
    regulators, rr, signs = prime2rr(primes[node])

    regulators_dict[node] = regulators
    rr_dict[node] = rr
    signs_dict[node] = signs

In [11]:
f = open(json_file)
json_dict = json.load(f)

constraints = json_dict["constraints"]

check = True
for node in primes:
    start = time.perf_counter()

    check = check_node(regulators_dict[node],
                       rr_dict[node],
                       rr_dict[node],
                       constraints,
                       node) and check

    end = time.perf_counter()
    print(f"Checking {node} took {end - start:.6f} seconds.")

Checking APC took 0.000037 seconds.
Checking BCL6 took 0.000065 seconds.
Checking CD28 took 0.000095 seconds.
Checking CD4 took 0.000009 seconds.
Checking CD8 took 0.000016 seconds.
Checking CMAF took 0.000019 seconds.
Checking DLL1 took 0.000004 seconds.
Checking EOMES took 0.000010 seconds.
FOXP3: regulator FOXP3 is completely absent
FOXP3: regulator RORGT is completely absent
Checking FOXP3 took 0.000348 seconds.
Checking GATA3 took 0.010150 seconds.
Checking GZMB took 0.000014 seconds.
Checking IFNAR took 0.000057 seconds.
Checking IFNA_e took 0.000002 seconds.
Checking IFNBR took 0.000011 seconds.
Checking IFNB_e took 0.000001 seconds.
Checking IFNG took 0.009365 seconds.
Checking IFNGR took 0.000040 seconds.
Checking IFNGR_2 took 0.000038 seconds.
Checking IFNG_2 took 0.021890 seconds.
Checking IFNG_e took 0.000007 seconds.
Checking IKB took 0.000100 seconds.
Checking IL10 took 0.000242 seconds.
Checking IL10R took 0.000098 seconds.
Checking IL10_e took 0.000005 seconds.
Checking